In [1]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from PIL import Image
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import requests


In [ ]:

# 1. Load a pre-trained Mask R-CNN model
print("Loading model...")
weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
model = maskrcnn_resnet50_fpn(weights=weights)
model.eval() # Set to evaluation mode

# Get the category names from the weights
COCO_INSTANCE_CATEGORY_NAMES = weights.meta["categories"]

# 2. Get a test image
img_url = "http://images.cocodataset.org/val2017/000000039769.jpg" # Image with cats
raw_image = Image.open(requests.get(img_url, stream=True).raw).convert("RGB")
img_tensor = F.to_tensor(raw_image)

# 3. Run the model (inference)
print("Running inference...")
with torch.no_grad():
    prediction = model([img_tensor])

# 4. Extract and filter results (masks, boxes, labels)
# We'll only keep predictions with a confidence score > 0.5
pred = prediction[0]
keep = pred['scores'] > 0.5
filtered_boxes = pred['boxes'][keep]
filtered_labels = pred['labels'][keep]
filtered_masks = pred['masks'][keep]

print(f"Found {len(filtered_labels)} objects with score > 0.5")

# 5. Visualize and Save
fig, ax = plt.subplots(1, figsize=(12, 9))
ax.imshow(raw_image)

for i in range(len(filtered_labels)):
    # Get details for this instance
    mask = filtered_masks[i, 0].cpu().numpy()
    label_index = filtered_labels[i].item()
    label_name = COCO_INSTANCE_CATEGORY_NAMES[label_index - 1] # COCO is 1-indexed
    box = filtered_boxes[i].cpu().numpy()

    print(f"- Object {i+1}: {label_name}")

    # Draw the mask
    color = np.random.rand(3) # Random color for each mask
    alpha = 0.5
    colored_mask = np.zeros((mask.shape[0], mask.shape[1], 3))
    colored_mask[mask > 0.5] = color
    
    ax.imshow(colored_mask, alpha=alpha)

    # Draw the bounding box
    rect = patches.Rectangle(
        (box[0], box[1]), box[2] - box[0], box[3] - box[1],
        linewidth=2, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)
    
    # Add the label
    ax.text(box[0], box[1] - 5, label_name, color=color,
            bbox=dict(facecolor='white', alpha=0.7, pad=0.5))

ax.axis('off')
output_filename = "maskrcnn_output.png"
plt.savefig(output_filename, bbox_inches='tight')
print(f"\nDone! Saved output to {output_filename}")